# DWthon Tools - Dzień 3: `Robot przygotowuje. Ty decydujesz.`

Dziś dowozimy publiczną obietnicę tego wydarzenia w najbardziej dosłownej formie: **Poranny Briefing**, który czyta kontrolowane dane, buduje trzy priorytety, przygotowuje follow-up i **zatrzymuje się** dokładnie tam, gdzie zaczyna się Twoja decyzja.

Wczoraj zbudowaliśmy silnik. Dziś opakowujemy go w produkt MVP, który można pokazać zespołowi, klientowi albo sobie rano przed pierwszym spotkaniem - i od razu poczuć, że to nie jest kolejne demo z LinkedIna.

> #### Autonomia nie jest przełącznikiem "AI on/off".
> #### Jest drabiną, na którą wchodzi się na danych - i z której można zejść w każdej chwili.

> Gradio nie robi produktu. Gradio pokazuje stan, który już istnieje. Jeśli stan jest bałaganem, dostaniesz ładny bałagan.

## Gdzie jesteśmy: pętla domknięta

Pamiętasz pętlę z Dnia 1? Dziś stawiamy na niej ostatnie dwa klocki.

```text
1. Ty    → model: katalog (enum) + schema decyzji (Pydantic)       ✅ Dzień 1
2. model → Ty:    obiekt ToolCall                                  ✅ Dzień 1
3. Ty    → Ty:    POLICY: czy wolno? argument ugruntowany?         ✅ Dzień 2
4. Ty    → świat: ROUTER (dict + if, limity z policy)              ✅ Dzień 2
5. świat → Ty:    surowy wynik                                     ✅ Dzień 2
6. Ty    → Ty:    NORMALIZACJA → evidence                          ✅ Dzień 2
7. Ty    → model: obserwacja + schema następnej decyzji            ✅ Dzień 2
8.        → HUMAN GATE + AUDIT + PRODUKT                           🔨 DZIŚ
```

| # | Warstwa | Dzień 1 | Dzień 2 | Dzień 3 |
| --- | --- | --- | --- | --- |
| 1 | 🗂️ Capability Registry | enum | 3 zdolności + adapter | licznik zdolności |
| 2 | 🎯 Decision | `ToolCall` | decyzja z evidence | - |
| 3 | 🚧 Policy / Guard | 3 koszyki | `is_allowed` blokuje | policy jako pokrętło w UI |
| 4 | ⚙️ Router | atrapa | `ROUTER` + awarie | log wywołań na ekranie |
| 5 | 🧼 Normalization | raw vs clean | `EvidenceItem` | panel "skąd to wiesz?" |
| 6 | 🧑‍⚖️ Human Gate | 3 koszyki | `ActionPreview` | przycisk, którego robot nie kliknie |
| 7 | 🧾 Audit / Trace | - | `reason` w obiekcie | pełny ślad decyzji |

Dwie ostatnie warstwy nie są dodatkiem UX. To one decydują, czy ktokolwiek pozwoli agentowi dotknąć prawdziwej skrzynki.

## Czego nauczysz się dzisiaj

1. Domkniesz **warstwę 6**: `ActionPreview` stanie się decyzją człowieka, a nie akcją systemu.
2. Domkniesz **warstwę 7**: zobaczysz, co system widział, zaproponował, zablokował i co zrobił człowiek.
3. Zobaczysz, że funkcja `needs_human` z Zadania 2.3 steruje przyciskiem w produkcie.
4. Uruchomisz ten sam poranek w trybie `KONTROLOWANY` i `NAIWNY` - różnicę zobaczysz w liczbach.
5. Sprawdzisz odporność na cztery ataki i awarie jednym kliknięciem.
6. Zrozumiesz różnicę między `approval-required` a `auto-eligible`.
7. Zobaczysz, dlaczego notebook, `helper.py` i `app.py` muszą być trzema plikami.


Zwróć uwagę, że zabierzesz wzorzec, który przeniesiesz na kalendarz, Jirę, bazę, repo i faktury - zmieniając tylko router.

## Plan na dzisiaj

```text
      08:15 - "uruchom poranek"
         │
         ▼
┌────────────────────────────────────────────────────────────────┐
│ 1 KATALOG    3 zdolności read-only. Nic poza tym nie istnieje  │
│ 2 DECYZJA    structured output → ToolCall (Pydantic)           │
│ 3 POLICY     przepuść / zablokuj / obetnij + POWÓD             │
│ 4 ROUTER     dict + if. Limity z policy, nie z życzeń modelu   │
│ 5 NORMALIZ.  surowe dane → evidence. Podejrzane OZNACZONE      │
│ 6 HUMAN GATE ActionPreview: przygotowane, NIE wysłane          │
│ 7 AUDIT      ślad: co widział, co zablokował, kto zdecydował   │
└────────────────────────────────────────────────────────────────┘
         │
         ▼
   3 priorytety · 2 drafty · 1 propozycja akcji · 0 wysłanych maili
```

> **Cała trudna praca jest zrobiona. Ostatnia sekunda należy do Ciebie.**

```text
NARZĘDZIA TO NIE 200 TOOLI W KONTEKŚCIE. TO 7 WARSTW, KTÓRE PROJEKTUJESZ.

🗂️ REGISTRY      co w ogóle istnieje             [mały, świadomy]
🎯 DECISION      structured output, nie proza    [Pydantic]
🚧 POLICY        czy wolno i w jakim zakresie    [twarde nie]
⚙️ ROUTER        kto wykonuje i co przy awarii   [nudny = dobry]
🧼 NORMALIZ.     co wraca do kontekstu           [brama celna]
🧑‍⚖️ HUMAN GATE   co wymaga człowieka              [ostatnia sekunda]
🧾 AUDIT         dlaczego tak się stało          [zaufanie]

MCP daje Ci 1 i 4. Pozostałe pięć to Twój produkt.
```

## Trzy pliki, trzy odpowiedzialności

```text
NOTEBOOK (day3)          helper.py                 app.py
---------------          ---------                 ------
ORKIESTRATOR      ->     SILNIK              ->    INTERFEJS (Gradio)
"co uruchamiamy"        "jak to działa"           "jak to wygląda i klika"

- backend proof          - ToolName / ToolCall      - panele, przyciski, CSS
- odpala app.py          - BriefingPolicy           - na klik woła helper
- linki, instrukcje      - is_allowed / normalize   - render 7 warstw
                           - collect_evidence        - nie zna logiki policy
                           - build_briefing
```

Ta separacja ma dwie konsekwencje:

1. **Zmieniasz UI bez dotykania logiki.** Dziś Gradio, jutro FastAPI + React, Slack bot albo cron o 8:00. `helper.py` się nie rusza.
2. **Podmieniasz źródło danych bez przepisywania produktu.** `dwthon_google_fake` → `dwthon_google_real`. Jedna linia, reszta bez zmiany.

Jeśli podmiana adaptera wymaga zmian w UI, granica była fikcją.

## Dlaczego dziś `app.py`, a nie inline Gradio w komórce?

1. To ten sam wzorzec, którego używamy w Agentic AI.
2. `app.py` jest gotowym plikiem, więc notebook nie musi przepisywać interfejsu w komórkach.
3. Taki układ jest stabilniejszy w JupyterHub.
4. Unikamy problemu `uvicorn.run()` z aktywnym event loopem notebooka.
5. Dostajesz prawdziwą aplikację na porcie, a nie obiekt w komórce.
6. Notebook jest przewodnikiem, `app.py` jest interfejsem.
7. Po warsztacie wynosisz `app.py` poza notebook praktycznie bez zmiany kodu.

In [ ]:
import json
from pprint import pprint

from dwthon_google_fake import WorkspaceClient
import helper as h

## Dowód backendu, zanim zobaczymy UI

Zanim włączymy cokolwiek kolorowego: **silnik już dowozi produkt**. UI tylko go pokaże.

W komórce niżej robimy dokładnie to samo, co wczoraj, z jedną różnicą: policy jest jawnie ustawiona na `approval-required`. To znaczy: "robot przygotowuje, człowiek zatwierdza".

In [ ]:
policy = h.BriefingPolicy(
    max_results=3,
    autonomy_mode=h.AutonomyMode.APPROVAL_REQUIRED,
)

client = WorkspaceClient(profile="founder")
snapshot = client.morning_snapshot(
    gmail_query=policy.gmail_query,
    max_emails=policy.max_results,
)

briefing = h.build_briefing("founder", snapshot, policy)

print(h.briefing_markdown(briefing))
print("\nUżyte elementy kontekstu:", briefing.context_items_used)

In [ ]:
print("=== ACTION PREVIEW (warstwa 6) ===")
print(briefing.action_preview.model_dump_json(indent=2))

print("\n=== CO BY POSZŁO, GDYBY CZŁOWIEK KLIKNĄŁ ===")
pprint(briefing.action_preview.payload)

print("\nCzy cokolwiek zostało wysłane?", False)
print("Czy istnieje metoda, która mogłaby to wysłać?", False)

### Zwróć uwagę na dwie ostatnie linijki

Pierwsza mówi, że nic nie poszło. **Druga jest ważniejsza**: nie istnieje metoda, która mogłaby to zrobić.

"Nie wysłałem" to stan. "Nie mam czym wysłać" to **architektura**. `send_email` nie ma w `ToolName`, więc nie da się go nawet wyrazić jako decyzji.

> Najlepsza obrona przed niebezpieczną akcją to funkcja, której nie napisałeś.

## Nowość Dnia 3: decyzja też jest danymi

Mamy dowody (`EvidenceItem`) i propozycje (`ActionPreview`). Brakuje trzeciego typu, bez którego nie ma produkcji: **zapisu decyzji** (`AuditEntry`).

Za tydzień ktoś zapyta: "dlaczego robot zaproponował ten mail?" Albo: "dlaczego tego nie zaproponował?" Bez zapisu pozostaje wzruszenie ramion i zdanie "model tak uznał".

| Pole | Co zawiera |
| --- | --- |
| `at` | kiedy |
| `layer` | która warstwa blueprintu 1-7 |
| `event` | co się stało |
| `detail` | powód blokady, liczba znaków, ID evidence |
| `actor` | `system` czy `human` |

`actor` odpowiada na pytanie, które zada każdy audytor i prawnik: **kto podjął tę decyzję?** Audyt nie jest po to, żeby się tłumaczyć. Jest po to, aby szybko zaufać automatyzacjom, które na to zasługują, i równie szybko wyłączyć te, które nie zasługują.

![Audit Trail](../images/day3-audit-trail.svg)


In [ ]:
# Warstwa 3 zostawia ślad szczególnie wtedy, gdy odmawia.
known_threads = {thread["id"] for thread in snapshot["gmail"]["threads"]}

blocked = h.ToolCall(tool=h.ToolName.GET_THREAD, arguments={"thread_id": "invented_id"})
allowed, reason = h.is_allowed(blocked, policy, known_threads)

entry = h.AuditEntry(
    layer=3,
    event="tool_call_blocked",
    detail=f"{blocked.tool.value}({blocked.arguments}) → {reason}",
    actor="system",
)
print(entry.model_dump_json(indent=2))

## Ten sam poranek, dwa światy

Policzmy to przed UI. Symulujemy naiwny pipeline, jaki powstaje, kiedy "najpierw niech zadziała":

- szeroki zakres odczytu: 14 dni i 10 maili zamiast 3,
- zero normalizacji: cały response leci do kontekstu,
- zero policy: katalog zawiera też akcje zmieniające świat,
- zero human gate: co model zaproponuje, to system wykonuje.

Nikt nie projektuje tego złośliwie. Tak wychodzi, kiedy nie projektuje się granic.

In [ ]:
naive_snapshot = client.morning_snapshot(
    gmail_query="in:inbox newer_than:14d",
    max_emails=10,
)
naive_threads = [client.get_thread(t["id"], full=True) for t in naive_snapshot["gmail"]["threads"]]
naive_context = json.dumps({"snapshot": naive_snapshot, "threads": naive_threads}, ensure_ascii=False)

evidence, dropped = h.collect_evidence(snapshot, policy)
controlled_context = "\n".join(item.model_dump_json() for item in evidence)

hijack_attempts = sum(
    1
    for thread in naive_threads
    for msg in thread.get("thread", {}).get("messages", [])
    if h.normalize_text(msg.get("body", ""), max_chars=100_000)[1]
)

print(f"{'':<28}{'NAIWNY':>12}{'KONTROLOWANY':>16}")
print("-" * 56)
print(f"{'znaki w kontekście':<28}{len(naive_context):>12,}{len(controlled_context):>16,}")
print(f"{'tokeny (≈)':<28}{len(naive_context)//4:>12,}{len(controlled_context)//4:>16,}")
print(f"{'wnioski z evidence ID':<28}{'0':>12}{len(evidence):>16}")
print(f"{'treści z próbą przejęcia':<28}{hijack_attempts:>12}{hijack_attempts:>16}")
print(f"{'...z tego oflagowanych':<28}{'0':>12}{hijack_attempts:>16}")
print(f"{'akcji bez zgody człowieka':<28}{'1':>12}{'0':>16}")
print(f"{'wpisów audytowych':<28}{'0':>12}{'≥6':>16}")

### Popatrz na dwa ostatnie wiersze

Różnica w tokenach jest efektowna, ale prawdziwy problem jest niżej. W obu światach istnieje ta sama próba przejęcia. Różnica jest w tym, że kontrolowany system **wie**, że ją widzi, oznacza ją i nie daje jej prawa do uzasadnienia akcji.

> **Naiwny pipeline nie jest mniej dopracowany. Jest niesterowalny. To dwie różne kategorie systemów, nie dwa poziomy jakości.**

## Autonomia nie jest przełącznikiem "AI on/off"

**`approval-required`** - robot przygotowuje, człowiek zatwierdza. To domyślny i najbardziej dojrzały tryb dla akcji zmieniających świat.

**`auto-eligible`** - automatycznie może przejść wyłącznie **klasa akcji** o znanym profilu ryzyka, nie "ten agent" albo "ta jedna akcja".

Kiedy wolno awansować klasę akcji?

1. Ma za sobą minimum kilkadziesiąt powtórzeń.
2. Ma 100% zatwierdzeń i zero edycji treści.
3. Jest odwracalna albo ma tani mechanizm cofnięcia.
4. Ma znany i mały blast radius.
5. Ma wyłącznik, który wraca do `approval` w jednym miejscu, bez deploya.

> **Kontrola nie jest hamulcem autonomii. Jest jej warunkiem. Bez audytu jedyna uczciwa autonomia to zero.**

W wersji warsztatowej oba tryby kończą się symulowanym wpisem audytowym. Nic nie trafia do Gmaila ani Calendar.

## 🆘 Zanim odpalisz serwer

1. Uruchom komórkę `!python app.py` i **nie zamykaj jej**.
2. Samo `[*]` nie wystarcza - proces może jeszcze się podnosić.
3. Do UI wejdź dopiero po `Application startup complete.` oraz `Uvicorn running on http://0.0.0.0:8080`.
4. Jeśli zobaczysz `Address already in use`, zatrzymaj starą komórkę przyciskiem Stop, poczekaj chwilę i uruchom ponownie.
5. Aplikacja jest osobnym procesem z własnym stanem w RAM. Restart komórki oznacza czysty start demo.
6. Jeśli UI się nie otwiera, sprawdź najpierw log startowy i ścieżkę `proxy/8080`.

In [ ]:
!python app.py

## Otwórz Poranny Briefing w nowej karcie

Gdy log potwierdzi start, kliknij:

## 👉👉 [Klik ⇒ PORANNY BRIEFING (Port 8080)](/hub/user-redirect/proxy/8080/) 👈👈

![](../images/day3-app-hero.jpg)

## Jak czytać ekran

1. **Pasek 7 warstw** pokazuje status blueprintu: przepuszczone, zablokowane, oflagowane albo wyłączone.
2. **KPI** streszczają warsztat. `maile wysłane przez robota` w trybie kontrolowanym zawsze pokazuje `0`.
3. **Budżet kontekstu** pokazuje raw chars, chars w kontekście, redukcję i tokeny.
4. **Briefing** daje trzy priorytety z `dlaczego teraz` i chipami evidence.
5. **Evidence** odpowiada na pytanie "skąd robot to wie?". Oflagowane źródła są widoczne, ale nie mają władzy.
6. **Human Gate** ma propozycję, powód, podgląd i decyzje człowieka. Nie ma przycisku "Wyślij".
7. **Audit Trail** pokazuje, co system widział, zablokował i kto zdecydował.
8. **Testy odporności** pozwalają jednym kliknięciem zepsuć agenta w cztery klasyczne sposoby.
9. **Pod maską** pokazuje policy JSON, raw vs normalized, komendę `gog`, obiekt Pydantic i eksport audytu.

## Co możesz zrobić?

1. Profil `founder`, tryb `KONTROLOWANY` → kliknij `Uruchom poranek`.
2. Obejrzyj Evidence. Oflagowane źródło ma być widoczne, ale nie powinno uzasadniać priorytetu.
3. Kliknij cztery testy odporności. Przy `send_email` zobaczysz `ValueError`, bo takiej zdolności nie da się zbudować jako obiektu.
4. Zjedź do Human Gate. Przeczytaj draft. Poszukaj przycisku "Wyślij". Nie ma go.
5. Przełącz na `NAIWNY` i uruchom ten sam poranek. Karty 3, 5, 6 i 7 gasną, kontekst rośnie, a symulacja pokazuje wysłany mail.
6. Wróć do `KONTROLOWANY`, zatwierdź propozycję i znajdź w Audit Trail `actor: human`.
7. Przełącz profil na `quiet_day`. Robot ma powiedzieć "dziś nie ma podstaw do tej akcji", zamiast wymyślać pracę.

![](../images/day3-app-naive-vs-controlled.svg)

## Self-check MVP

1. Czy potrafisz wskazać liczbę redukcji kontekstu i wyjaśnić, skąd się bierze?
2. Czy oflagowany wątek jest widoczny, ale nie uzasadnia akcji?
3. Czy `send_email` pokazuje `ValueError`, a nie odmowę modelu?
4. Czy każdy priorytet ma co najmniej jedno evidence ID?
5. Czy w trybie naiwnym warstwy 3, 5, 6 i 7 są wygaszone?
6. Czy po zatwierdzeniu w audycie pojawia się `actor: human`?
7. Czy na `quiet_day` robot nie wymyśla follow-upu?
8. Czy umiesz powiedzieć, jakie dane audytowe są potrzebne, aby awansować akcję na `auto-eligible`?

## 🧠 Zadanie 3.1: jedno pokrętło, cały system

Zmień **jeden** parametr policy, zbuduj briefing i porównaj wynik.

Dotykasz prawdziwej dźwigni produktu: policy zmienia zachowanie systemu bez przepisywania aplikacji. Spróbuj czegoś, co coś zepsuje: ustaw `max_body_chars=40` i sprawdź, czy briefing nadal ma sens. Granica ma chronić, ale nie może kastrować.

Ta liczba to decyzja produktowa, nie techniczna.

In [ ]:
# YOUR CODE HERE
my_policy = h.BriefingPolicy()  # zmień JEDEN parametr

my_snapshot = client.morning_snapshot(
    gmail_query=my_policy.gmail_query,
    max_emails=my_policy.max_results,
)
my_briefing = h.build_briefing("founder", my_snapshot, my_policy)
my_evidence, my_dropped = h.collect_evidence(my_snapshot, my_policy)

print({
    "context_items_used": my_briefing.context_items_used,
    "evidence_chars": sum(len(i.summary) for i in my_evidence),
    "dropped": my_dropped,
    "action_status": my_briefing.action_preview.status,
    "first_priority": my_briefing.priorities[0].title if my_briefing.priorities else None,
})

<details>
<summary>👉 Podpowiedź</summary>

Zmieniaj **po jednym**: `max_results`, `max_body_chars` albo `autonomy_mode`. Nie dotykaj silnika briefingu - o to chodzi w tym zadaniu.

<details>
<summary>👉 Trzy warianty warte przetestowania</summary>

```python
# A) skrajnie oszczędny kontekst
my_policy = h.BriefingPolicy(max_results=1, max_body_chars=40)

# B) wyższa autonomia
my_policy = h.BriefingPolicy(autonomy_mode=h.AutonomyMode.AUTO_ELIGIBLE)

# C) szerszy zakres odczytu
my_policy = h.BriefingPolicy(gmail_query="in:inbox newer_than:7d", max_results=8)
```

Wariant C jest najbardziej pouczający. Zmieniłeś jedno słowo w zapytaniu, a wpuściłeś agentowi tydzień korespondencji. Zapytanie do źródła danych jest elementem policy - traktuj je z taką samą powagą jak uprawnienia.

</details>
</details>

## 🧠 Zadanie 3.2: zaprojektuj swojego robota

To ostatnie i najważniejsze zadanie. Wypełnij szkielet dla własnego przypadku użycia. Trzy zdolności, nie dziesięć.

Dwa pola, których nie umiałeś wypełnić w Dniu 1, to `blast_radius` i `promotion_rule`. Od nich zaczyna się specyfikacja, którą można wdrożyć, zamiast samego pomysłu na agenta.

In [ ]:
# YOUR CODE HERE - artefakt, który zabierasz z warsztatu
my_robot = {
    "name": "...",
    "runs_when": "...",
    "data_scope": "...",  # zakres odczytu = policy. Bądź brutalnie wąski.
    "capabilities": ["...", "...", "..."],  # maks. 3, wyłącznie READ
    "never_in_catalog": ["..."],
    "human_gate": ["..."],
    "blast_radius": "...",
    "evidence": "...",
    "audit": "...",
    "promotion_rule": "...",
}

for key, value in my_robot.items():
    print(f"{key:>18} : {value}")

<details>
<summary>👉 Przykład wypełniony: robot wsparcia</summary>

```python
my_robot = {
    "name": "Robot: Przegląd zgłoszeń supportu",
    "runs_when": "codziennie 8:00 + na żądanie",
    "data_scope": "tickets: status=open AND updated_at > now()-24h AND queue='billing'",
    "capabilities": [
        "search_tickets(query)  # read-only",
        "get_ticket(ticket_id)  # tylko ID z search",
        "get_customer_plan(customer_id)  # bez danych płatniczych",
    ],
    "never_in_catalog": ["close_ticket", "refund", "reply_to_customer"],
    "human_gate": ["każda odpowiedź do klienta", "każda eskalacja", "oflagowane źródło"],
    "blast_radius": "1 klient / 1 ticket na akcję; brak akcji masowych",
    "evidence": "ticket_id + cytat 1-2 zdań + plan klienta",
    "audit": "kto zatwierdził, co poprawił, co zablokowano i dlaczego",
    "promotion_rule": "auto-eligible dla tagu priorytetu: ≥50 przypadków, 100% approve, 0 edycji, akcja odwracalna",
}
```

Nie ma tu ani jednej zdolności zapisu. Robot i tak zdejmuje najtrudniejszą część pracy: czytanie, rozumienie i pisanie. Człowiek dostaje ostatnią sekundę.

> To nie jest okrojona wersja agenta. To jest wersja, którą wdrożysz w poniedziałek.

</details>

<div class="alert alert-block alert-danger">
<h2>✅ Zaliczenie Dnia 3</h2>

1. 📸 Screenshot głównego widoku `KONTROLOWANY`: pasek 7 warstw, KPI i budżet kontekstu.
2. 📸 Screenshot Evidence z oflagowanym źródłem oraz priorytetem, który się na nim nie opiera.
3. 🧨 Uruchom cztery testy odporności.
4. 💬 Wrzuć rezultat na Slacka: `#dwthon_tools_day3_done` i dopisz, gdzie w Twoim robocie kończy się automatyzacja, a zaczyna decyzja człowieka.

Nie oceniamy, czy model napisał najpiękniejszy mail. Oceniamy, czy potrafisz wyjaśnić, **dlaczego system zaproponował właśnie ten krok i jak go zatrzymać**.
</div>


## Co jest prawdziwe, a co symulowane?

**Prawdziwy jest wzorzec.** Mały katalog zdolności, structured output jako decyzja, policy z powodem odmowy, deterministyczny router, normalizacja do evidence, human gate przed akcją i audyt z `actor`.

**Prawdziwy jest adapter.** `google_workspace_adapter.py` naprawdę woła `gog` z flagami `--readonly --gmail-no-send --no-input --json --wrap-untrusted`.

**Symulowane są dane**, aby każdy miał ten sam poranek i żeby można było powtarzalnie testować próbę przejęcia.

**Symulowana jest akcja zewnętrzna.** `Zatwierdź` zapisuje wpis do audytu i nie dotyka Gmaila. To świadome MVP.

**Symulowany jest tryb naiwny.** Nie chcemy, aby najważniejsza pointa warsztatu zależała od humoru modelu danego dnia.

Gdy zechcesz pracować na własnym koncie: zmieniasz import na `dwthon_google_real`, konfigurujesz `gog` poza warsztatem i zachowujesz tę samą policy.

## Co masz zrobione w trzy dni?

| |  |  |
| --- | --- | --- |
| narzędzia | "trzeba podłączyć MCP" | 7 warstw, z których protokół daje 2 |
| tool calling | magia z `tools=` | enum + Pydantic + `dict` + `if` |
| bezpieczeństwo | akapit w system prompcie | funkcja, której nie napisałeś |
| kontekst | cały response API | evidence z ID |
| cudza treść | tekst w prompcie | dane bez władzy |
| akcja | "wyśle maila" | propozycja z ostatnią sekundą dla Ciebie |
| autonomia | przełącznik | drabina z regułą awansu opartą na audycie |
| artefakt | slajd | działający produkt na porcie 8080 |

Ani razu nie potrzebowałeś `tools=`, frameworka agentowego ani serwera MCP. Nie dlatego, że są złe. Dlatego, że teraz wiesz, co jest pod nimi i co nadal musisz zbudować sam.

## Podsumowanie całego DWthon Tools

1. **MCP to kabel, nie mózg.** JSON-RPC i trzy wywołania. Świetny standard połączenia, zerowa odpowiedzialność za decyzję.
2. **Model nie ma rąk - ma usta.** Zwraca obiekt, wykonuje Twój kod.
3. **Bezpieczeństwo siedzi w reprezentacji, nie w prośbie.** `send_email` nie da się wyrazić, bo nie ma go w enumie.
4. **Wynik narzędzia to jeszcze nie kontekst.** Normalizacja jest warunkiem trafności.
5. **Evidence to cytat z numerem.** Bez ID nie ma odpowiedzi na "skąd to wiesz?".
6. **Nieufna treść ma głos, ale nie ma władzy.** Oznaczamy, nie cenzurujemy.
7. **Human gate to nie tryb dla tchórzy.** Daje dane potrzebne do zwiększania autonomii.
8. **Audyt jest warunkiem autonomii, nie biurokracją.**
9. **Zmienia się router. Nie zmienia się architektura.** Kalendarz, Jira, baza, repo, faktury - ten sam szkielet.

```text
❌ Tools ≠ MCP       ❌ Tools ≠ Function calling      ❌ Tools ≠ Integracja
❌ Tools ≠ Wykonanie ❌ Tools ≠ Autonomia              ❌ Tools ≠ Prompt
❌ Tools ≠ Katalog
```

## Gdzie ten wzorzec idzie dalej

**1. Więcej zdolności, ta sama architektura.** Dokładasz wpis w routerze, policy i regułę human gate. Trzydzieści narzędzi wymaga kuracji katalogu, nie innej architektury.

**2. MCP jako transport, nie jako mózg.** Klient MCP ląduje pod routerem. Katalog nadal filtrujesz Ty, argumenty waliduje policy, wynik normalizuje warstwa 5.

**3. Planowanie adaptacyjne.** Agent planujący sekwencję, zmieniający plan po błędzie i wiedzący, kiedy przestać, to następny poziom. Bez warstw 3, 6 i 7 jest tylko szybszym sposobem robienia szkód.

> **Nie przeskakujesz poziomów. Agent, który planuje, ale nie ma policy, audytu i granicy człowieka, to nie wyższy poziom autonomii. To ten sam poziom, tylko z większą prędkością.**

## Co dalej?

DWthon Tools to **czwarty z pięciu DWthonów**. Każdy pozwala wejść w temat, zbudować coś działającego i sprawdzić, czy taki sposób pracy z AI chcesz rozwijać dalej.

<div style="text-align: center;">
  <a href="https://pay.dataworkshop.eu/?cart=%7B%22products%22%3A%7B%22212%22%3A1%7D%2C%22discountCode%22%3A%22season_pass_founding%22%7D&lang=pl&currency=pln&gate=pl"><img src="../images/season-pass-tools.svg" width="900" alt="Season Pass"></a>
</div>

- [**Season Pass / cała ścieżka 5 DWthonów**](https://pay.dataworkshop.eu/?cart=%7B%22products%22%3A%7B%22212%22%3A1%7D%2C%22discountCode%22%3A%22season_pass_founding%22%7D&lang=pl&currency=pln&gate=pl)
- [**Praktyczny LLM**](https://dataworkshop.eu/pl/praktyczny-llm)
- [**Agentic AI**](https://dataworkshop.eu/agentic-ai)

Dzięki za Twoją aktywność przez te trzy dni.

## Przydatne linki

1. [Model Context Protocol](https://modelcontextprotocol.io) · [ogłoszenie Anthropic](https://www.anthropic.com/news/model-context-protocol)
2. [Anthropic - Building effective agents](https://www.anthropic.com/engineering/building-effective-agents)
3. [Anthropic - Writing effective tools for AI agents](https://www.anthropic.com/engineering/writing-tools-for-agents)
4. [Anthropic - Code execution with MCP](https://www.anthropic.com/engineering/code-execution-with-mcp)
5. [Simon Willison - The lethal trifecta](https://simonwillison.net/2025/Jun/16/the-lethal-trifecta/)
6. [gog - Google Workspace in your terminal](https://gogcli.sh)
7. [instructor](https://python.useinstructor.com/) · [Pydantic validators](https://docs.pydantic.dev/latest/concepts/validators/)
8. [Gradio docs](https://www.gradio.app/docs)
9. [OWASP Top 10 for LLM Applications](https://owasp.org/www-project-top-10-for-large-language-model-applications/)